# 📧 Email Marketing — Campaign & Lead Management

**Manages campaigns, email templates, and lead-to-campaign assignments via Notion.**

| Section | Functions | Databases |
|---------|-----------|-----------|
| 1 — Read | `read_campaigns()`, `read_templates()`, `read_leads()` | campaigns, email_sequences, leads_crm |
| 2 — Campaigns | `create_campaign()`, `activate_campaign()`, `pause_campaign()` | campaigns |
| 3 — Templates | `create_template()`, `list_templates_for_campaign()` | email_sequences |
| 4 — Lead Assignment | `assign_leads_by_score()`, `preview_assignments()`, `clear_assignments()` | leads_crm |

### Property Access

| Database | Read + Write | Read Only |
|----------|-------------|-----------|
| **campaigns** | Name, Status, Send Frequency, Priority, Goal, Start Date, Notes | Campaign Score, Open Rate, Click Rate, Total Sent, Bounce Rate, Reply Rate |
| **email_sequences** | Template, Subject Line, Campaign, Status, Trigger, Send Delay Hours, Notes | Open Rate, Click Rate, Unsubscribe Rate, Total Sent, Bounce Rate, Reply Rate |
| **leads_crm** | Audience relation only | Email, Name, First Name, Score, Segment, Status, Unsubscribed, Pipeline Stage, Last Touch, Source, Signals |

> **Note:** Email sending is handled by [Email_send.ipynb](../Email_send.ipynb). This notebook manages the data layer only.

In [1]:
# ── Cell 1: Setup & Configuration ────────────────────────────────────
import sys, os

_WS_ROOT = r"C:\Users\sossi\Desktop\Business\Orchestrator Hedge Edge"
_MARKETING_DIR = os.path.join(_WS_ROOT, "Business", "GROWTH", "executions", "Marketing")

# Ensure both workspace root and marketing package parent are importable
for p in [_WS_ROOT, _MARKETING_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

from dotenv import load_dotenv
load_dotenv(os.path.join(_WS_ROOT, ".env"))

from shared.notion_client import DATABASES
import shared.notion_client as _nc

# ── Fix: shared.notion_client._extract_value drops relation properties ──
# The function returns None for type="relation" because there's no handler.
# Monkey-patch it to return a list of page IDs.
_original_extract = _nc._extract_value
def _patched_extract(prop):
    if prop.get("type") == "relation":
        return [r["id"] for r in prop.get("relation", [])]
    return _original_extract(prop)
_nc._extract_value = _patched_extract
print("  Patched _extract_value to handle relation properties")

# ── Import execution modules (all real logic lives here) ──
from email_marketing.campaigns import read_campaigns, create_campaign, activate_campaign, pause_campaign
from email_marketing.templates import read_templates, create_template, list_templates_for_campaign
from email_marketing.leads import read_leads, preview_assignments, assign_leads_by_score, clear_assignments
from email_marketing.leads import _read_leads_full
from email_marketing.config import SEGMENTS

print("Setup complete")
print(f"  NOTION_API_KEY set: {bool(os.getenv('NOTION_API_KEY') or os.getenv('NOTION_TOKEN'))}")
print(f"  Campaigns DB:       {DATABASES['campaigns'][:8]}...")
print(f"  Email Sequences DB: {DATABASES['email_sequences'][:8]}...")
print(f"  Leads CRM DB:       {DATABASES['email_sends'][:8]}...")
print(f"  Functions imported:  campaigns(4) templates(3) leads(4)")

  Patched _extract_value to handle relation properties
Setup complete
  NOTION_API_KEY set: True
  Campaigns DB:       38566e9d...
  Email Sequences DB: 2258cf16...
  Leads CRM DB:       30b652ea...
  Functions imported:  campaigns(4) templates(3) leads(4)


## Section 1 — Notion Read

Read-only queries against all three databases. Returns clean Python dicts with both writable and read-only fields.

In [2]:
# ── Section 1: Read Functions ─────────────────────────────────────────
# Imported from execution/campaigns.py, execution/templates.py, execution/leads.py
#   read_campaigns(status_filter=None)
#   read_templates(campaign_id=None)
#   read_leads(segment=None, exclude_unsubscribed=True)

print("Read functions imported: read_campaigns(), read_templates(), read_leads()")

Read functions imported: read_campaigns(), read_templates(), read_leads()


In [3]:
# ── Section 1 Demo: Read all three databases ─────────────────────────

# Campaigns
campaigns = read_campaigns()
print(f"=== CAMPAIGNS ({len(campaigns)}) ===")
for c in campaigns:
    metrics = f"score={c['campaign_score']}  open={c['open_rate']}  click={c['click_rate']}  bounce={c['bounce_rate']}  reply={c['reply_rate']}"
    print(f"  [{c['status']:>15}] {c['name']:<35} freq={c['send_frequency']}h  {metrics}")

# Templates (all)
templates = read_templates()
print(f"\n=== EMAIL TEMPLATES ({len(templates)}) ===")
for t in templates:
    metrics = f"open={t['open_rate']}  click={t['click_rate']}  bounce={t['bounce_rate']}  reply={t['reply_rate']}"
    print(f"  {t['template_name']:<40} subject: {t['subject_line'][:40]}  {metrics}")

# Leads (sample — first 10)
leads = read_leads()
print(f"\n=== LEADS ({len(leads)} total, showing first 10) ===")
for lead in leads[:10]:
    print(f"  [{lead['segment']:>4}] {lead['email']:<35} score={lead['score']:<4} stage={lead['pipeline_stage']}  src={lead['source']}")

=== CAMPAIGNS (6) ===
  [         Active] Hedge Awareness Campaign            freq=36h  score=50  open=26  click=4  bounce=1  reply=0
  [         Active] Re-Engagement Campaign              freq=72h  score=80  open=0  click=0  bounce=0  reply=0
  [In building phase] The Hedge Report Newsletter         freq=336h  score=10  open=0  click=0  bounce=0  reply=0
  [         Active] Warm Lead Conversion                freq=72h  score=70  open=49  click=9  bounce=0  reply=0
  [         Active] hot-lead follow up                  freq=48h  score=90  open=89  click=35  bounce=0  reply=0
  [         Active] Beta Activation                     freq=48h  score=90  open=0  click=0  bounce=0  reply=0

=== EMAIL TEMPLATES (18) ===
  Email 7                                  subject: Your founding member invite is ready  open=24  click=6  bounce=0  reply=0
  Email 6                                  subject: I failed 6 challenges before I figured t  open=18  click=4  bounce=1  reply=0
  Email 1          

## Section 2 — Campaign Management

Create, activate, and pause campaigns. Writes to: **Name, Status, Send Frequency, Priority, Goal, Start Date, Notes**.

In [4]:
# ── Section 2: Campaign Management Functions ─────────────────────────
# Imported from execution/campaigns.py
#   create_campaign(name, send_frequency=48, priority="Medium", goal="", notes="")
#   activate_campaign(campaign_id, campaign_name="")
#   pause_campaign(campaign_id, campaign_name="")

print("Campaign functions imported: create_campaign(), activate_campaign(), pause_campaign()")

Campaign functions imported: create_campaign(), activate_campaign(), pause_campaign()


In [5]:
# ── Section 2 Demo: Campaign lifecycle ────────────────────────────────
# Uncomment to test (writes to Notion):

# new = create_campaign(
#     name="Q1 Prop Trading Nurture",
#     send_frequency=72,
#     goal="Convert warm leads from prop trading segment",
#     campaign_score=50,
# )

# activate_campaign(new["id"], new["name"])
# pause_campaign(new["id"], new["name"])

# Verify current campaigns
print("Current campaigns:")
for c in read_campaigns():
    print(f"  [{c['status']:>15}] {c['name']}")

Current campaigns:
  [         Active] Hedge Awareness Campaign
  [         Active] Re-Engagement Campaign
  [In building phase] The Hedge Report Newsletter
  [         Active] Warm Lead Conversion
  [         Active] hot-lead follow up
  [         Active] Beta Activation


## Section 3 — Email Templates

Create email templates (sequences) linked to campaigns. Writes to: **Template, Subject Line, Campaign (relation), Status, Trigger, Send Delay Hours, Notes**.

In [6]:
# ── Section 3: Email Template Functions ───────────────────────────────
# Imported from execution/templates.py
#   create_template(campaign_id, template_name, subject_line, ...)
#   list_templates_for_campaign(campaign_id)

print("Template functions imported: create_template(), list_templates_for_campaign()")

Template functions imported: create_template(), list_templates_for_campaign()


In [7]:
# ── Section 3 Demo: List templates per active campaign ────────────────

active = read_campaigns(status_filter="Active")
if active:
    for camp in active:
        cid = camp["id"]
        print(f"\nTemplates for '{camp['name']}':")
        tpls = list_templates_for_campaign(cid)
        if tpls:
            for t in tpls:
                print(f"  #{t['_seq_num']}  {t['template_name']:<35} subject: {t['subject_line'][:50]}")
                print(f"       open={t['open_rate']}  click={t['click_rate']}  bounce={t['bounce_rate']}  reply={t['reply_rate']}")
        else:
            print("  (no templates)")
else:
    print("No active campaigns — create and activate one in Section 2 first.")


Templates for 'Hedge Awareness Campaign':
  #1  Email 1                             subject: How much have you actually spent on prop firm chal
       open=25  click=1  bounce=1  reply=1
  #2  Email 2                             subject: Your prop firm doesn't want you to know this
       open=32  click=4  bounce=2  reply=0
  #3  Email 3                             subject: We want you more than Trump wants Greenland
       open=33  click=12  bounce=0  reply=0
  #4  Email 4                             subject: The house always wins, unless you play both sides 
       open=20  click=0  bounce=0  reply=0
  #5  Email 5                             subject: The strategy prop firms hope you never Google
       open=27  click=2  bounce=1  reply=0
  #6  Email 6                             subject: I failed 6 challenges before I figured this out
       open=18  click=4  bounce=1  reply=0
  #7  Email 7                             subject: Your founding member invite is ready
       open=24  cli

## Section 4 — Lead Assignment

Assign leads to campaigns via the `Related to Email Campaigns (Audience)` dual-relation. This is the **only** writable property on leads_crm.

Score thresholds: **Cold** (0–29), **Warm** (30–59), **Hot** (60+)

In [8]:
# ── Section 4: Lead Assignment Functions ──────────────────────────────
# Imported from execution/leads.py
#   preview_assignments(campaign_id, min_score=0, segment=None)
#   assign_leads_by_score(campaign_id, min_score=0, segment=None, dry_run=True)
#   clear_assignments(campaign_id, dry_run=True)

print("Lead functions imported: preview_assignments(), assign_leads_by_score(), clear_assignments()")

Lead functions imported: preview_assignments(), assign_leads_by_score(), clear_assignments()


In [9]:
# ── Section 4 Demo: Which leads are in which campaigns? ───────────────
# Full view: load all leads with campaign relations, cross-reference campaign names

leads_all, email_to_rel = _read_leads_full(exclude_unsubscribed=True)
campaigns_all = read_campaigns()
camp_id_to_name = {c["id"]: c["name"] for c in campaigns_all}

# Build per-campaign lead lists
campaign_leads = {}
unassigned = []
for lead in leads_all:
    rel_ids = email_to_rel.get(lead["email"], [])
    if not rel_ids:
        unassigned.append(lead)
    else:
        for cid in rel_ids:
            cname = camp_id_to_name.get(cid, cid[:8] + "...")
            campaign_leads.setdefault(cname, []).append(lead)

print("=" * 70)
print("  LEAD → CAMPAIGN ASSIGNMENTS")
print("=" * 70)
for cname, members in sorted(campaign_leads.items(), key=lambda x: -len(x[1])):
    print(f"\n  📧 {cname}  ({len(members)} leads)")
    for m in members[:5]:
        print(f"      {m['email']:<35} [{m['segment']}] score={m['score']}")
    if len(members) > 5:
        print(f"      ... and {len(members) - 5} more")

print(f"\n  ⚠️  Unassigned leads: {len(unassigned)}")
for m in unassigned[:5]:
    print(f"      {m['email']:<35} [{m['segment']}] score={m['score']}")
if len(unassigned) > 5:
    print(f"      ... and {len(unassigned) - 5} more")

# Summary
total_assigned = sum(len(v) for v in campaign_leads.values())
print(f"\n{'─' * 70}")
print(f"  Total leads: {len(leads_all)} | Assigned: {total_assigned} | Unassigned: {len(unassigned)}")

  LEAD → CAMPAIGN ASSIGNMENTS

  📧 Hedge Awareness Campaign  (66 leads)
      leonkazungu5@gmail.com              [Cold] score=2
      adeshthosani76@gmail.com            [Cold] score=2
      muhammed.olafusi@gmail.com          [Cold] score=2
      diegogaliachoo@gmail.com            [Cold] score=1
      vasu9167@gmail.com                  [Cold] score=1
      ... and 61 more

  📧 Warm Lead Conversion  (64 leads)
      rickhjlee@gmail.com                 [Warm] score=7
      ryanghasry84@gmail.com              [Warm] score=6
      hoangvanhieu.hvnh@gmail.com         [Warm] score=7
      christianmoodie2@gmail.com          [Warm] score=3
      drnahuelfinance@gmail.com           [Warm] score=6
      ... and 59 more

  📧 hot-lead follow up  (5 leads)
      aldenlee@synergy.com.sg             [Hot] score=27
      oasishalifax@hotmail.com            [Hot] score=23
      mizraubaid1190@gmail.com            [Hot] score=24
      mdsahil1120@gmail.com               [Hot] score=25
      jamiefx

In [11]:
# ── Section 5: What emails need sending next? ─────────────────────────
# For each active campaign, show the next template in the sequence
# and which leads are eligible to receive it.

from datetime import datetime, timezone, timedelta

active_camps = read_campaigns(status_filter="Active")
leads_all2, email_to_rel2 = _read_leads_full(exclude_unsubscribed=True)

def _parse_last_send(val):
    """Extract an ISO string from last_send (may be str, dict, or None)."""
    if not val:
        return ""
    if isinstance(val, dict):
        return val.get("start") or ""
    return str(val)

print("=" * 70)
print("  NEXT EMAILS TO SEND")
print("=" * 70)

total_pending = 0
for camp in sorted(active_camps, key=lambda c: -c["campaign_score"]):
    cid = camp["id"]
    freq_hours = camp["send_frequency"]
    tpls = list_templates_for_campaign(cid)
    
    # Find leads assigned to this campaign
    assigned = [l for l in leads_all2 if cid in email_to_rel2.get(l["email"], [])]
    
    if not assigned:
        print(f"\n  {camp['name']}  (score={camp['campaign_score']}, freq={freq_hours}h)")
        print(f"      No leads assigned - skipping")
        continue
    
    # Check send eligibility: leads whose last_send was > freq_hours ago (or never)
    now = datetime.now(timezone.utc)
    eligible = []
    for lead in assigned:
        last = _parse_last_send(lead.get("last_send"))
        if not last:
            eligible.append(lead)
            continue
        try:
            last_dt = datetime.fromisoformat(last.replace("Z", "+00:00"))
            if (now - last_dt) >= timedelta(hours=freq_hours):
                eligible.append(lead)
        except (ValueError, TypeError):
            eligible.append(lead)
    
    print(f"\n  {camp['name']}  (score={camp['campaign_score']}, freq={freq_hours}h)")
    print(f"      Templates: {len(tpls)} | Assigned: {len(assigned)} | Eligible now: {len(eligible)}")
    
    if tpls:
        print(f"      Sequence:")
        for t in tpls:
            print(f"        #{t['_seq_num']} {t['template_name']:<30} -> \"{t['subject_line'][:45]}\"")
    
    if eligible:
        print(f"      Eligible leads (first 5):")
        for e in eligible[:5]:
            ls = _parse_last_send(e['last_send'])
            last_str = ls[:10] if ls else 'never'
            print(f"        {e['email']:<35} [{e['segment']}] score={e['score']}  last_send={last_str}")
        if len(eligible) > 5:
            print(f"        ... and {len(eligible) - 5} more")
    
    total_pending += len(eligible)

print(f"\n{'=' * 70}")
print(f"  TOTAL EMAILS PENDING: {total_pending} leads eligible across {len(active_camps)} active campaigns")

  NEXT EMAILS TO SEND

  hot-lead follow up  (score=90, freq=48h)
      Templates: 3 | Assigned: 5 | Eligible now: 5
      Sequence:
        #1 Hot lead email 1               -> "Quick one — want me to map this to your prop "
        #2 Hot lead email 2               -> "Most traders lose this at the execution step"
        #3 Hot lead email 3               -> "Should I close your file for now?"
      Eligible leads (first 5):
        aldenlee@synergy.com.sg             [Hot] score=27  last_send=2026-03-09
        oasishalifax@hotmail.com            [Hot] score=23  last_send=2026-03-18
        mizraubaid1190@gmail.com            [Hot] score=24  last_send=2026-03-12
        mdsahil1120@gmail.com               [Hot] score=25  last_send=2026-03-12
        jamiefxinvesting@gmail.com          [Hot] score=36  last_send=2026-03-12

  Beta Activation  (score=90, freq=48h)
      Templates: 3 | Assigned: 3 | Eligible now: 2
      Sequence:
        #1 Beta 1                         -> "Your Hedge